# 03 - Build Deployment Candidate Application

Build and validate a deployment-ready AI app with professional packaging and dependency management.


## Section 1: Deployment Candidate Definition

### Definition
A deployment candidate is app version that passed local quality gates and is safe to release.

### Theory
Candidate includes source code, tests, dependency lock, configuration templates, and operational docs.

### Motivation
Deploying ad-hoc prototype code creates downtime and debugging chaos.

### Real-World Example
Product team demo scheduled tomorrow. Candidate process prevents last-minute runtime failures.

### Visual Explanation
![Deployment Architecture](../outputs/figures/deployment-architecture.png)

### Best Practices
- Keep app entrypoint explicit (`app.py`).
- Use modular structure (`pages`, `components`, `utils`).
- Add fallback logic for external inference APIs.

### Common Mistakes
- Coupling UI and model code in one large script.
- No fallback when API quota/rate-limit hits.


In [1]:
from pathlib import Path

root = Path.cwd()
if not (root / "streamlit_app").exists() and (root.parent / "streamlit_app").exists():
    root = root.parent

for path in [
    root / "app.py",
    root / "streamlit_app" / "app.py",
    root / "streamlit_app" / "pages",
    root / "streamlit_app" / "utils" / "models.py",
    root / "tests",
]:
    print(path.relative_to(root), "->", "OK" if path.exists() else "MISSING")


app.py -> OK
streamlit_app/app.py -> OK
streamlit_app/pages -> OK
streamlit_app/utils/models.py -> OK
tests -> OK


## Section 2: Model Choice Rationale

### Definition
Model selection balances quality, latency, cost, and deployability.

### Theory
This project uses three-tier inference strategy:
1. Hugging Face hosted models for cloud runtime.
2. Ollama local models for local experimentation.
3. Rule-based fallback for guaranteed response.

### Motivation
Free-tier hosting may not support large GPU models. Fallback keeps app available.

### Real-World Example
If external API times out, app still returns output via fallback instead of error page.

### Visual Explanation
![Fallback Strategy](../outputs/figures/fallback-strategy.png)

### Best Practices
- Expose inference method in UI for transparency.
- Log latency and fallback usage.

### Common Mistakes
- Silent fallback that hides degraded behavior from maintainers.


## Section 3: Dependency Management Deep Dive

### Definition
- `requirements.txt`: plain list used by many hosts.
- `pyproject.toml`: modern project metadata and dependency declaration.
- `uv.lock`: exact resolved versions for reproducibility.

### Theory
Use `pyproject.toml` + `uv.lock` as source of truth, then export `requirements.txt` for deployment target compatibility.

### Motivation
Reproducible dependencies prevent surprise breakages during cloud build.

### Real-World Example
Unpinned transitive package releases breaking change and deployment fails.

### Best Practices
- Lock dependencies.
- Keep runtime and dev tooling explicit.
- Regenerate lock file after dependency changes.

### Common Mistakes
- Editing requirements manually without syncing lock.
- Mixing `pip`, `conda`, and `uv` in same project.


In [2]:
from pathlib import Path
import tomllib

root = Path.cwd()
if not (root / "pyproject.toml").exists() and (root.parent / "pyproject.toml").exists():
    root = root.parent

pyproject = tomllib.loads((root / "pyproject.toml").read_text())
deps = pyproject["project"].get("dependencies", [])
print("Dependency count:", len(deps))
print("First five dependencies:")
for dep in deps[:5]:
    print("-", dep)

print("uv.lock exists:", (root / "uv.lock").exists())
print("requirements.txt exists:", (root / "requirements.txt").exists())


Dependency count: 4
First five dependencies:
- streamlit>=1.58.0
- requests>=2.34.0
- numpy>=2.5.0
- pandas>=3.0.0
uv.lock exists: True
requirements.txt exists: True


## Section 4: Local Testing Before Deployment

### Definition
Local validation ensures app starts, inference works, and tests pass before cloud push.

### Theory
Fail fast locally, not in production.

### Motivation
Cloud debugging loops are slower and harder due remote logs and limited shell access.

### Practical Checklist
1. Create venv and sync deps.
2. Run test suite.
3. Launch Streamlit app.
4. Verify each user flow.
5. Confirm secrets are externalized.

### Common Mistakes
- Skipping tests before pushing to main.
- Testing only one page and deploying all pages.


In [3]:
from pathlib import Path
import subprocess

root = Path.cwd()
if not (root / "tests").exists() and (root.parent / "tests").exists():
    root = root.parent

cmd = ["uv", "run", "pytest", "-q"]
result = subprocess.run(cmd, cwd=root, capture_output=True, text=True, check=False)
print("Command:", " ".join(cmd))
print("Exit code:", result.returncode)
print("Pytest summary:")
lines = (result.stdout + "\n" + result.stderr).splitlines()
for line in lines[-6:]:
    print(line)


Command: uv run pytest -q
Exit code: 0
Pytest summary:
........................................................................ [ 67%]
..................................                                       [100%]
106 passed in 0.86s

